# Feature Engineering Analysis

This notebook demonstrates the feature engineering pipeline for the age and gender prediction project.
The actual implementation is in `src/features/` for reproducibility.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image

# Add src to path for imports
sys.path.append('../src')

from src.features.build_features import load_processed_splits, build_datasets
from src.features.preprocessing import preprocess_facial_image
from src.config import PROJECT_ROOT, IMAGE_SIZE, NUM_CHANNELS

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Load Processed Data

Load the train/test splits created by `make_dataset.py`

In [ ]:
# Load processed splits
train_df, test_df = load_processed_splits()

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nTraining data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

# Display first few rows
train_df.head()

## 2. Feature Distribution Analysis

In [ ]:
# Age and gender distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Age distribution
axes[0].hist(train_df['age'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0].set_title('Age Distribution (Training Set)')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')

# Gender distribution
gender_counts = train_df['gender'].value_counts()
axes[1].bar(gender_counts.index, gender_counts.values, color=['lightcoral', 'lightblue'])
axes[1].set_title('Gender Distribution (Training Set)')
axes[1].set_xlabel('Gender (0=Male, 1=Female)')
axes[1].set_ylabel('Count')

# Age vs Gender boxplot
sns.boxplot(data=train_df, x='gender', y='age', ax=axes[2])
axes[2].set_title('Age Distribution by Gender')
axes[2].set_xlabel('Gender (0=Male, 1=Female)')
axes[2].set_ylabel('Age')

plt.tight_layout()
plt.show()

# Statistics
print("\n=== Dataset Statistics ===")
print(f"Age range: {train_df['age'].min()} - {train_df['age'].max()} years")
print(f"Mean age: {train_df['age'].mean():.1f} years")
print(f"Gender balance: {gender_counts[0]} males, {gender_counts[1]} females")
print(f"Gender ratio: {gender_counts[1]/gender_counts[0]:.2f} (F/M)")

## 3. Image Preprocessing Pipeline

Demonstrate the preprocessing steps applied to facial images

In [ ]:
# Select a sample image for demonstration
sample_path = train_df.iloc[0]['image_path']
full_path = os.path.join(PROJECT_ROOT, sample_path)

print(f"Sample image: {sample_path}")
print(f"Age: {train_df.iloc[0]['age']}, Gender: {train_df.iloc[0]['gender']}")

# Load and display original image
original_img = Image.open(full_path)
print(f"Original size: {original_img.size}")

# Apply preprocessing pipeline
processed_tensor = preprocess_facial_image(full_path, apply_equalization=True, final_normalization='[-1, 1]')
processed_img = processed_tensor.numpy()

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original image
axes[0].imshow(original_img)
axes[0].set_title(f'Original Image\n{original_img.size}')
axes[0].axis('off')

# Processed image (grayscale, resized)
if NUM_CHANNELS == 1:
    axes[1].imshow(processed_img.squeeze(), cmap='gray')
else:
    axes[1].imshow(processed_img)
axes[1].set_title(f'Processed Image\n{IMAGE_SIZE}, Normalized [-1,1]')
axes[1].axis('off')

# Pixel value distribution
axes[2].hist(processed_img.flatten(), bins=50, alpha=0.7, color='green')
axes[2].set_title('Pixel Value Distribution\n(After Normalization)')
axes[2].set_xlabel('Pixel Value')
axes[2].set_ylabel('Frequency')
axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.7, label='Zero')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\nProcessed tensor shape: {processed_tensor.shape}")
print(f"Pixel value range: [{processed_img.min():.3f}, {processed_img.max():.3f}]")

## 4. Preprocessing Steps Analysis

In [ ]:
# Compare different preprocessing options
sample_paths = train_df['image_path'].head(4).tolist()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

for i, rel_path in enumerate(sample_paths):
    full_path = os.path.join(PROJECT_ROOT, rel_path)
    
    # Original
    original = Image.open(full_path)
    axes[i, 0].imshow(original)
    axes[i, 0].set_title(f'Original {i+1}')
    axes[i, 0].axis('off')
    
    # Without histogram equalization
    proc_no_eq = preprocess_facial_image(full_path, apply_equalization=False, final_normalization='[0, 1]')
    if NUM_CHANNELS == 1:
        axes[i, 1].imshow(proc_no_eq.numpy().squeeze(), cmap='gray')
    else:
        axes[i, 1].imshow(proc_no_eq.numpy())
    axes[i, 1].set_title('No Equalization')
    axes[i, 1].axis('off')
    
    # With histogram equalization
    proc_with_eq = preprocess_facial_image(full_path, apply_equalization=True, final_normalization='[0, 1]')
    if NUM_CHANNELS == 1:
        axes[i, 2].imshow(proc_with_eq.numpy().squeeze(), cmap='gray')
    else:
        axes[i, 2].imshow(proc_with_eq.numpy())
    axes[i, 2].set_title('With Equalization')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## 5. TensorFlow Dataset Pipeline

Test the complete feature engineering pipeline

In [ ]:
# Build TensorFlow datasets
print("Building TensorFlow datasets...")
ds_train, ds_test, train_df_full, test_df_full = build_datasets(batch_size=32)

print(f"Training dataset: {ds_train}")
print(f"Test dataset: {ds_test}")

# Inspect a batch
for batch_images, (batch_ages, batch_genders) in ds_train.take(1):
    print(f"\nBatch shapes:")
    print(f"Images: {batch_images.shape}")
    print(f"Ages: {batch_ages.shape}")
    print(f"Genders: {batch_genders.shape}")
    
    print(f"\nSample values:")
    print(f"Age range in batch: [{batch_ages.numpy().min():.1f}, {batch_ages.numpy().max():.1f}]")
    print(f"Gender distribution: {np.bincount(batch_genders.numpy().astype(int))}")
    print(f"Image pixel range: [{batch_images.numpy().min():.3f}, {batch_images.numpy().max():.3f}]")

## 6. Feature Engineering Summary

### Preprocessing Pipeline:
1. **Image Loading**: Read JPEG files using TensorFlow
2. **Color Conversion**: Convert to grayscale (1 channel)
3. **Resizing**: Resize to 360×360 pixels using bilinear interpolation
4. **Normalization**: Scale pixel values to [-1, 1] range
5. **Histogram Equalization**: Optional contrast enhancement

### Key Features:
- **Reproducible**: All preprocessing in `src/features/preprocessing.py`
- **Configurable**: Parameters in `src/config.py`
- **Efficient**: TensorFlow dataset pipeline with caching and prefetching
- **Quality Control**: Consistent image dimensions and value ranges

### Next Steps:
- Model architecture design
- Training pipeline implementation
- Hyperparameter optimization

In [ ]:
# Final validation
print("=== Feature Engineering Pipeline Validation ===")
print(f"✓ Data loaded: {len(train_df)} train, {len(test_df)} test samples")
print(f"✓ Image preprocessing: {IMAGE_SIZE} grayscale, [-1,1] normalized")
print(f"✓ TensorFlow datasets: Batched, cached, and prefetched")
print(f"✓ Age range: {train_df['age'].min()}-{train_df['age'].max()} years")
print(f"✓ Gender balance: {train_df['gender'].value_counts().to_dict()}")
print("\n🎉 Feature engineering pipeline ready for model training!")